## Algorithm:  BERT-based Transformer for classifying Fake News


Dataset: [LIAR Dataset]

https://huggingface.co/datasets/liar

Ref Dataset Paper :https://arxiv.org/abs/1705.00648


In [ ]:
!pip install transformers
!pip install datasets
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 21.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0


# 1: Load the dataset

The LIAR dataset is a benchmark dataset for detecting fabricated news. It contains 12,836 labeled short political statements compiled from Politifact, a fact-checking web site. All statements are annotated into one of six levels of truthfulness:

"pants-fire" (completely false)

"false"

"barely-true"

"half-true"

"mostly-true"

"true"


In [ ]:
print("Loading the LIAR dataset...")
dataset = load_dataset("liar", trust_remote_code=True)
dataset = dataset.rename_column("label", "labels")

Loading the LIAR dataset...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.16k [00:00<?, ?B/s]

liar.py:   0%|          | 0.00/6.41k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10269 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1283 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1284 [00:00<?, ? examples/s]

#  2: Load a pretrained BERT tokenizer

In [ ]:

print("Loading the BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Loading the BERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

#  3: Tokenize the text

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["statement"], padding="max_length", truncation=True)

print("Tokenizing dataset...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Tokenizing dataset...


Map:   0%|          | 0/10269 [00:00<?, ? examples/s]

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

#  4: Load BERT model (for **6-class classification**)

In [ ]:
print("Loading the BERT model...")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=6)


Loading the BERT model...


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



#  5: Define an evaluation function

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted",zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

#  6: Set training arguments

In [ ]:
#  6: Set training arguments
training_args = TrainingArguments(
    output_dir="./results",
    run_name="fake_news_bert_experiment",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,  # Keeps the best version of the model
    metric_for_best_model="f1",
    report_to='none'
)

#  7: Initialize Trainer

In [ ]:
import os

os.environ["WANDB_DISABLED"] = "true"


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)


# Training and Evalution

In [ ]:
#  8: Train the model
print("Starting training...")
trainer.train()

#  9: Evaluate the model
print("Evaluating the model...")
trainer.evaluate()


Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.260800,2.063118,0.235981,0.251018,0.235981,0.227488
2,0.859000,2.451596,0.251558,0.257262,0.251558,0.239026
3,0.746400,2.900211,0.245327,0.253179,0.245327,0.243784
4,0.409600,3.577487,0.255452,0.265937,0.255452,0.249779
5,0.126800,4.418734,0.255452,0.262929,0.255452,0.248868
6,0.109500,5.177073,0.254673,0.261092,0.254673,0.251116
7,0.132800,5.783558,0.260125,0.264197,0.260125,0.256022
8,0.060500,5.953994,0.252336,0.255888,0.252336,0.250116


Evaluating the model...


{'eval_loss': 5.783558368682861,
 'eval_accuracy': 0.2601246105919003,
 'eval_precision': 0.26419696149267935,
 'eval_recall': 0.2601246105919003,
 'eval_f1': 0.25602218027333423,
 'eval_runtime': 36.3895,
 'eval_samples_per_second': 35.285,
 'eval_steps_per_second': 2.226,
 'epoch': 8.0}

##### Observations:
  - There is potential for improvement by adding external knowledge sources or using larger transformer models.


##### ***Conclusion***:
  - Fine-tuned BERT model on LIAR dataset.
-  Carried out preprocessing through tokenization and encoding of the labels.
-   Evaluating the model's performance with a set of evaluation metrics.
- Observed where the model had to get better, such as support for uncertain sentences and data augmentation.

